# Segmentación no supervisada de perfiles financieros

Comparamos tres métodos para encontrar perfiles semejantes: **K-Prototypes**, **One-Hot + K-Means** y **Gower + K-Medoids**. No hay una columna objetivo y los clusters no asignan automáticamente una ruta de crédito.

> La base es sintética. Los grupos reflejan la forma en que se generaron los registros; no representan una muestra observada de estudiantes ni clientes reales.

## Criterio de variables

Excluimos `id_persona`, `rango_edad`, ingreso reportado bruto, `tiene_ingresos`, los rubros individuales de gastos, `nivel_educacion_financiera` y `flujo_mensual_disponible_cop`. Usamos edad en lugar del rango; ingreso neto en lugar del ingreso bruto; gasto esencial total en lugar de sus componentes. Dejamos el proxy de educación financiera y el flujo para perfilar los clusters después, sin duplicar sus variables de origen dentro de la distancia.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
PROCESSED_REL = Path("data/processed/perfiles_crediticios_procesados.csv")
if not (ROOT / PROCESSED_REL).exists() and (ROOT.parent / PROCESSED_REL).exists():
    ROOT = ROOT.parent
if not (ROOT / PROCESSED_REL).exists():
    ROOT = Path("/content/Modelo")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/catalinagaviriamet-spec/Modelo-de-Inclusi-n-Crediticia-Estudiantes-UdeM.git", str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(ROOT))
from src.models.segmentacion import comparar_clusterizaciones, CATEGORICAS

perfiles = pd.read_csv(ROOT / PROCESSED_REL)
metricas, etiquetas, elegido, etiquetas_por_modelo = comparar_clusterizaciones(
    perfiles, k_values=(3, 4, 5, 6), seed=42, gower_train_size=1800, evaluation_size=800
)
print(f"Perfiles: {len(perfiles):,} | método con mejor silhouette Gower de evaluación: {elegido}")
display(metricas.round(3))

## Cómo comparar los resultados

El `silhouette_gower` se calcula sobre la misma muestra de evaluación y la misma distancia mixta para los tres métodos. También observamos cuánto ocupa el cluster más pequeño y el más grande. Un silhouette alto no basta para elegir: los grupos deben tener tamaños útiles, perfiles comprensibles y estabilidad. Gower + K-Medoids se ajusta sobre una muestra por el coste cuadrático de calcular distancias por pares.

In [ ]:
perfilado = perfiles.copy()
perfilado["cluster"] = etiquetas.to_numpy()
numericas_resumen = [
    "edad", "personas_hogar", "personas_a_cargo", "ingreso_neto_mensual_cop",
    "gastos_esenciales_mensuales_cop", "cuotas_deudas_vigentes_cop",
    "ahorro_disponible_cop", "meses_historial_crediticio",
    "moras_ultimos_12_meses", "monto_solicitado_cop",
    "flujo_mensual_disponible_cop",
]
resumen_clusters = perfilado.groupby("cluster").agg(
    personas=("id_persona", "count"),
    edad_mediana=("edad", "median"),
    ingreso_neto_mediano_cop=("ingreso_neto_mensual_cop", "median"),
    gasto_esencial_mediano_cop=("gastos_esenciales_mensuales_cop", "median"),
    cuotas_deuda_mediana_cop=("cuotas_deudas_vigentes_cop", "median"),
    flujo_mediano_cop=("flujo_mensual_disponible_cop", "median"),
    meses_historial_mediana=("meses_historial_crediticio", "median"),
).round(0)
resumen_clusters["porcentaje"] = (resumen_clusters["personas"] / len(perfilado) * 100).round(1)
display(resumen_clusters)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
resumen_clusters["personas"].plot.bar(ax=axes[0], color="#4C78A8")
axes[0].set(title=f"Tamaño de clusters — {elegido[0]}, k={elegido[1]}", xlabel="Cluster", ylabel="Perfiles")
perfilado.boxplot(column="flujo_mensual_disponible_cop", by="cluster", ax=axes[1], grid=False)
axes[1].set(title="Flujo disponible por cluster", xlabel="Cluster", ylabel="COP")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
for columna in CATEGORICAS:
    print(f"Distribución de {columna} por cluster (proporciones):")
    display(pd.crosstab(perfilado["cluster"], perfilado[columna], normalize="index").round(2))

print("Proxy de educación financiera por cluster (solo para interpretación):")
display(pd.crosstab(perfilado["cluster"], perfilado["nivel_educacion_financiera"], normalize="index").round(2))

## Lectura y límites

El nombre `cluster` es solo un identificador técnico. Después de revisar medianas y proporciones, el equipo puede describir los perfiles con nombres neutrales. No usamos estas agrupaciones para aprobar préstamos. La variable de educación financiera es un proxy del historial crediticio; el flujo disponible es un insumo, no un monto aprobado. Ambos deben validarse antes de traducir segmentos a rutas de atención.

Los resultados se guardan en `data/processed/comparacion_clusterizaciones.csv` y `data/processed/perfiles_clusterizados.csv` al ejecutar `python scripts/entrenar_clusterizaciones.py`.